# Phishing URL Safety Checker

**Student:** Amit  
**Program:** IBM SkillsBuild Cyber Security BootCamp 2026

## Project overview
This beginner-friendly project checks a URL for a few common warning signs that can appear in phishing links. It gives a risk score and a simple result to help a user decide whether a link needs more checking.

**Important:** This is an educational demo, not a full security product. A low score does not prove that a website is safe. Do not open suspicious links just to test them.


## How it works
The program checks for signs such as an IP address used as the host, an unusually long URL, an `@` symbol, a URL shortener, selected suspicious-looking words, or a non-HTTPS scheme. Each sign adds points. The final label is based on the total points.

These are simple learning rules. Real phishing detection needs more context and stronger methods.


In [ ]:
from urllib.parse import urlparse
import ipaddress

URL_SHORTENERS = {"bit.ly", "tinyurl.com", "t.co", "is.gd", "ow.ly", "buff.ly"}
SUSPICIOUS_WORDS = {
    "login", "verify", "update", "secure", "account", "password",
    "payment", "bank", "confirm", "urgent", "free", "gift"
}

def check_url(url):
    """Return a simple risk score, label, and reasons for a URL."""
    url = url.strip()
    if not url:
        return {"url": url, "score": 0, "label": "Invalid input",
                "reasons": ["Please enter a URL."]}

    candidate = url if "://" in url else "https://" + url
    parsed = urlparse(candidate)
    host = (parsed.hostname or "").lower()
    reasons = []
    score = 0

    if not host:
        return {"url": url, "score": 0, "label": "Invalid input",
                "reasons": ["The URL host could not be read."]}

    if parsed.scheme.lower() != "https":
        score += 1
        reasons.append("The link does not use HTTPS.")

    if len(url) > 75:
        score += 1
        reasons.append("The URL is unusually long.")

    if "@" in url:
        score += 2
        reasons.append("The URL contains an @ symbol, which can confuse readers.")

    try:
        ipaddress.ip_address(host)
        score += 2
        reasons.append("The host is an IP address rather than a regular domain name.")
    except ValueError:
        pass

    if host in URL_SHORTENERS:
        score += 1
        reasons.append("The domain is a URL shortener, so the final destination may be hidden.")

    found_words = sorted(word for word in SUSPICIOUS_WORDS if word in url.lower())
    if found_words:
        score += 1
        reasons.append("The URL includes account-related or attention-grabbing words: "
                       + ", ".join(found_words) + ".")

    if host.count("-") >= 2:
        score += 1
        reasons.append("The domain contains several hyphens.")

    if score >= 4:
        label = "High risk — check carefully"
    elif score >= 2:
        label = "Some warning signs"
    else:
        label = "Few warning signs found"

    if not reasons:
        reasons.append("No warning signs from this small rule set were found.")

    return {"url": url, "score": score, "label": label, "reasons": reasons}

print("URL checker is ready.")


## Try the checker
These examples are for demonstration only. This is not a live reputation check, and the labels are not proof that a site is safe or malicious.


In [ ]:
sample_urls = [
    "https://www.wikipedia.org",
    "http://192.0.2.10/login/verify-account",
    "https://bit.ly/example",
    "http://account-update-confirm.example.com/urgent-login",
    "https://www.python.org"
]

for item in sample_urls:
    result = check_url(item)
    print("-" * 60)
    print("URL:", result["url"])
    print("Risk score:", result["score"])
    print("Result:", result["label"])
    print("Reasons:")
    for reason in result["reasons"]:
        print("  -", reason)


## Test your own URL
Paste a URL as text below. The checker only analyses the text; it does not visit the website.


In [ ]:
user_url = input("Enter a URL to check (do not open it): ")
result = check_url(user_url)
print("\nResult:", result["label"])
print("Risk score:", result["score"])
for reason in result["reasons"]:
    print("-", reason)


## Limitations and possible improvements
- The checker uses a short list of hand-written rules and can produce false positives or miss phishing links.
- It does not check domain age, certificates, redirects, page content, or live threat-intelligence feeds.
- HTTPS alone does not mean a website is trustworthy.
- A future version could use a verified public dataset, measure precision and recall, and compare this rule-based method with a machine-learning model.

## Conclusion
This project demonstrates a basic cybersecurity idea: look for suspicious URL patterns before trusting a link. Avoid unexpected links, verify the sender through another channel, and use an organisation-approved security tool when available.
